# 01 — Data Exploration & Dataset Acquisition (Pre-2000 YFCC15M Supplement)

This copy is configured to supplement **Timeguessr AI** with **pre-2000 geotagged photographs (`1900 <= year <= 1999`)** from `mehdidc/yfcc15m`:
1. **Local Timeguessr daily challenges** (`data/images/Timeguessr/`)
2. **OSV-5M** ([`osv5m/osv5m`](https://huggingface.co/datasets/osv5m/osv5m)) — global street-view geolocation images
3. **PastVu** ([`nyuuzyou/pastvu`](https://huggingface.co/datasets/nyuuzyou/pastvu)) — historical geotagged & year-labeled photographs
4. **YFCC15M (Pre-2000)** ([`mehdidc/yfcc15m`](https://huggingface.co/datasets/mehdidc/yfcc15m)) — filtered strictly for **`1900 <= year <= 1999`** + valid geolocation, saved to `data/images/yfcc15m_pre2000/`

In [1]:
import torch

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch: 2.11.0+cu128 | CUDA available: True
Device: NVIDIA GeForce RTX 4070 SUPER
VRAM: 12.88 GB


## 0. Configuration & Directories

These datasets are very large at full scale (OSV-5M is ~250 GB, PastVu is ~2M images across 2,094 tar shards, and YFCC15M's metadata CSV alone is ~10 GB). Adjust the limits below to start with a manageable subset or set them to `None` to download everything.

> **Optional:** If you hit Hugging Face rate limits, set your `HF_TOKEN` in a `.env` file or call `huggingface_hub.login()`.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login, get_token, whoami

load_dotenv()

# --- Hugging Face Authentication (raises Hub/resolver rate limits) ---
hf_token = os.getenv("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print(f"Authenticated with Hugging Face as: {whoami()['name']}")
elif get_token():
    print(f"Authenticated with cached HF token as: {whoami()['name']}")
else:
    print("Not authenticated with Hugging Face (using lower anonymous rate limits).")
    print("Set HF_TOKEN in .env or uncomment login() below to authenticate interactively:")
    # login()

# Base data directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = DATA_DIR / "images"

OSV5M_DIR = IMAGES_DIR / "osv5m"
PASTVU_DIR = IMAGES_DIR / "pastvu"
YFCC_DIR = IMAGES_DIR / "yfcc15m_pre2000"  # Separate folder so existing data/images/yfcc15m/ is preserved
TIMEGUESSR_DIR = IMAGES_DIR / "Timeguessr"

for d in (OSV5M_DIR, PASTVU_DIR, YFCC_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Global Year Bounds (Timeguessr range) ---
MIN_YEAR = 1900
MAX_YEAR = 2026

# PastVu: number of valid (1900-2026, geotagged) images to stream and save
PASTVU_MAX_SAMPLES = 50_000  # None = all valid images (1900-2026, geotagged)

# --- Balanced Sampling Controls ---
# alpha = 0.0 -> strictly uniform | 0.5 -> square-root (sublinear) | 1.0 -> raw distribution
OSV5M_TARGET_SAMPLES = 50_000
OSV5M_COUNTRY_ALPHA = 0.6
OSV5M_MAX_PER_SEQUENCE = 2
OSV5M_MAX_ZIPS = 33          # More shards = better coverage of rare countries (set None for all 98)

# YFCC15M (Pre-2000): scan strictly for 1900 <= year <= 1999
YFCC_MIN_YEAR = 1900
YFCC_MAX_YEAR = 1999              # Strictly before year 2000
YFCC_MIN_GEO_ACCURACY = 11        # Flickr accuracy: 1=world, 3=country, 6=region, 11=city, 16=street
YFCC_DOWNLOAD_WORKERS = 32
YFCC_TARGET_IMAGES = 15_000
YFCC_COUNTRY_ALPHA = 0.6
YFCC_MAX_VALID_SCAN = 30_000      # Valid pre-2000 metadata rows to collect (or full stream if fewer)
YFCC_MAX_TOTAL_SCAN = None        # Optional cap on raw rows scanned (None = scan entire 15M stream if needed)
YFCC_OVERSAMPLE_FACTOR = 2.0      # Extra metadata per bucket to survive dead Flickr URLs
YFCC_GEOCODE_BATCH_SIZE = 1_000

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

Authenticated with cached HF token as: darksabre1500
Project root: d:\VSCodeProjects\CS441
Data dir:     d:\VSCodeProjects\CS441\data


## 3. Download & Filter `mehdidc/yfcc15m` (Pre-2000 Geotagged Photos Only)

`mehdidc/yfcc15m` is a ~10 GB metadata CSV (`yfcc15m.csv`) from YFCC100M. We stream it and keep **only** historical rows strictly before the year 2000 (`1900 <= year <= 1999`) that satisfy strict geolocation and date quality checks:
- **Date checks (fast year pre-filter first to skip post-2000 rows quickly):**
  - `datetaken` is present and parses cleanly as a real calendar date
  - Year is strictly within `YFCC_MIN_YEAR..YFCC_MAX_YEAR` (`1900..1999`)
  - Excludes common uncalibrated camera/epoch defaults (`1900-01-01`, `1904-01-01`, `1970-01-01`, `1980-01-01`, `2000-01-01`, `0000-00-00`)
  - If `dateuploaded` is present, verifies `datetaken <= dateuploaded + 1 day`
- **Geolocation checks:**
  - `latitude` and `longitude` are present, finite floats within `[-90, 90]` and `[-180, 180]`
  - Not null-island `(0.0, 0.0)`
  - `accuracy >= YFCC_MIN_GEO_ACCURACY` (when present; filters out world/country-level centroids)
  - Must have a non-empty `downloadurl` so we can fetch the photo

After saving the filtered metadata to `data/images/yfcc15m_pre2000/metadata_filtered.csv`, the second cell downloads the actual image files concurrently into `data/images/yfcc15m_pre2000/images/` and validates each image with Pillow.

In [3]:
import pandas as pd

def allocate_sublinear_quotas(counts: dict, total_target: int, alpha: float = 0.5) -> dict:
    """Allocate `total_target` items across keys with weights proportional to count**alpha,
    capped at each key's available count (water-filling redistribution)."""
    avail = {k: int(v) for k, v in counts.items() if v > 0}
    if not avail or total_target <= 0:
        return {}
    if sum(avail.values()) <= total_target:
        return avail.copy()

    quotas = {k: 0 for k in avail}
    remaining = total_target
    active = dict(avail)

    while remaining > 0 and active:
        weights = {k: (v ** alpha) for k, v in active.items()}
        w_sum = sum(weights.values())
        allocated_this_round = 0
        saturated = []

        for k, cap in active.items():
            share = int(remaining * (weights[k] / w_sum))
            add = min(share, cap - quotas[k])
            quotas[k] += add
            allocated_this_round += add
            if quotas[k] >= cap:
                saturated.append(k)

        remaining -= allocated_this_round
        for k in saturated:
            active.pop(k, None)

        # Distribute rounding remainder 1-by-1 to highest-weight unsaturated keys
        if allocated_this_round == 0 and remaining > 0 and active:
            for k in sorted(active, key=lambda x: active[x] ** alpha, reverse=True):
                if remaining == 0:
                    break
                if quotas[k] < active[k]:
                    quotas[k] += 1
                    remaining -= 1
            break

    return {k: q for k, q in quotas.items() if q > 0}


def build_country_year_quotas(df: pd.DataFrame, total_target: int, country_alpha: float = 0.5) -> dict[tuple[str, int], int]:
    """Sublinear across countries, then uniform (alpha=0.0) water-filling across years within each country."""
    country_counts = df["country"].value_counts().to_dict()
    country_quotas = allocate_sublinear_quotas(country_counts, total_target, alpha=country_alpha)

    cell_quotas: dict[tuple[str, int], int] = {}
    grouped = df.groupby("country")["year"].value_counts()
    for country, c_quota in country_quotas.items():
        yr_counts = grouped.loc[country].to_dict()
        yr_quotas = allocate_sublinear_quotas(yr_counts, c_quota, alpha=0.0)
        for yr, q in yr_quotas.items():
            cell_quotas[(country, int(yr))] = q
    return cell_quotas

In [4]:
import math
from datetime import datetime, timezone, timedelta
import pandas as pd
import reverse_geocoder as rg
from datasets import load_dataset
from tqdm.auto import tqdm

BAD_CAMERA_TIMESTAMPS = {
    "1900-01-01 00:00:00",
    "1904-01-01 00:00:00",
    "1970-01-01 00:00:00",
    "1980-01-01 00:00:00",
    "2000-01-01 00:00:00",
    "0000-00-00 00:00:00",
}

def _parse_yfcc_datetime(raw) -> datetime | None:
    if raw is None:
        return None
    s = str(raw).strip()
    if not s or s.lower() in ("null", "none", "nan") or s in BAD_CAMERA_TIMESTAMPS:
        return None
    s_clean = s.split(".")[0]
    for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
        try:
            return datetime.strptime(s_clean, fmt)
        except ValueError:
            continue
    return None

def _parse_yfcc_uploaded(raw) -> datetime | None:
    if raw is None:
        return None
    s = str(raw).strip()
    if not s or s.lower() in ("null", "none", "nan"):
        return None
    try:
        ts = float(s)
        if ts > 0:
            return datetime.fromtimestamp(ts, tz=timezone.utc).replace(tzinfo=None)
    except ValueError:
        pass
    return _parse_yfcc_datetime(s)

def _flush_geocode_batch(batch: list[dict], out_list: list[dict]) -> None:
    if not batch:
        return
    coords = [(r["latitude"], r["longitude"]) for r in batch]
    # mode=1 avoids multiprocessing spawn issues on Windows notebooks
    results = rg.search(coords, mode=1)
    for rec, geo in zip(batch, results):
        rec["country"] = geo.get("cc", "")
        out_list.append(rec)
    batch.clear()

def stream_and_balance_yfcc15m(
    output_dir: Path = YFCC_DIR,
    max_valid_scan: int = YFCC_MAX_VALID_SCAN,
    max_total_scan: int | None = YFCC_MAX_TOTAL_SCAN,
    target_images: int = YFCC_TARGET_IMAGES,
    country_alpha: float = YFCC_COUNTRY_ALPHA,
    oversample_factor: float = YFCC_OVERSAMPLE_FACTOR,
    min_year: int = YFCC_MIN_YEAR,
    max_year: int = YFCC_MAX_YEAR,
    min_accuracy: int = YFCC_MIN_GEO_ACCURACY,
    geocode_batch_size: int = YFCC_GEOCODE_BATCH_SIZE,
    reuse_existing: bool = True,
) -> tuple[pd.DataFrame, dict[tuple[str, int], int]]:
    meta_path = output_dir / "metadata_filtered.csv"

    if reuse_existing and meta_path.exists():
        existing_df = pd.read_csv(meta_path, dtype={"photoid": str})
        if len(existing_df) > 0 and "bucket_target" in existing_df.columns:
            quotas = {
                (str(c), int(y)): int(t)
                for (c, y), t in existing_df.groupby(["country", "year"])["bucket_target"].first().items()
            }
            print(f"Reusing {len(existing_df):,} pre-2000 candidate rows from {meta_path} "
                  f"(target={sum(quotas.values()):,} images). Set reuse_existing=False to re-stream.")
            return existing_df, quotas

    ds = load_dataset("mehdidc/yfcc15m", split="train", streaming=True)

    valid_pool: list[dict] = []
    batch: list[dict] = []
    scanned = 0
    pbar = tqdm(total=max_valid_scan, desc=f"YFCC15M valid rows ({min_year}-{max_year})")

    try:
        for row in ds:
            scanned += 1
            if scanned % 50_000 == 0:
                pbar.set_postfix(scanned=f"{scanned:,}")
            if max_total_scan is not None and scanned > max_total_scan:
                break

            # Fast year pre-filter before float/datetime parsing (rejects >99% of post-2000 rows immediately)
            raw_dt = row.get("datetaken")
            if not raw_dt:
                continue
            s_dt = str(raw_dt).strip()
            if len(s_dt) < 4 or not s_dt[:4].isdigit():
                continue
            yr_fast = int(s_dt[:4])
            if not (min_year <= yr_fast <= max_year):
                continue

            url = row.get("downloadurl")
            if not url or not isinstance(url, str) or not url.startswith("http"):
                continue

            try:
                lat = float(row.get("latitude"))
                lon = float(row.get("longitude"))
            except (TypeError, ValueError):
                continue
            if not (math.isfinite(lat) and math.isfinite(lon)):
                continue
            if not (-90.0 <= lat <= 90.0 and -180.0 <= lon <= 180.0) or (lat == 0.0 and lon == 0.0):
                continue

            acc = row.get("accuracy")
            if acc is not None and str(acc).strip() not in ("", "null", "None", "nan"):
                try:
                    acc_val = int(float(acc))
                    if acc_val < min_accuracy:
                        continue
                except (TypeError, ValueError):
                    acc_val = None
            else:
                acc_val = None

            dt_taken = _parse_yfcc_datetime(raw_dt)
            if dt_taken is None or not (min_year <= dt_taken.year <= max_year):
                continue

            dt_uploaded = _parse_yfcc_uploaded(row.get("dateuploaded"))
            if dt_uploaded is not None and dt_taken > (dt_uploaded + timedelta(days=1)):
                continue

            batch.append({
                "photoid": str(row.get("photoid")),
                "year": dt_taken.year,
                "datetaken": dt_taken.isoformat(sep=" "),
                "latitude": lat,
                "longitude": lon,
                "accuracy": acc_val,
                "downloadurl": url,
                "title": row.get("title", ""),
                "source": "yfcc15m_pre2000",
            })
            pbar.update(1)

            if len(batch) >= geocode_batch_size:
                _flush_geocode_batch(batch, valid_pool)

            if len(valid_pool) + len(batch) >= max_valid_scan:
                break
    except KeyboardInterrupt:
        print("\nInterrupted! Geocoding buffered rows and building balanced metadata from scanned pool...")
    finally:
        _flush_geocode_batch(batch, valid_pool)
        pbar.close()

    if not valid_pool:
        print(f"No valid YFCC15M rows collected for {min_year}-{max_year}.")
        return pd.DataFrame(), {}

    pool_df = pd.DataFrame(valid_pool)
    pool_df = pool_df[pool_df["country"].astype(str).str.len() == 2].copy()
    print(f"Scanned {scanned:,} rows -> {len(pool_df):,} valid pre-2000 geotagged+dated rows "
          f"({int(pool_df['year'].min())}-{int(pool_df['year'].max())}) across {pool_df['country'].nunique()} countries.")

    # Compute exact download quotas per (country, year), then oversample candidates for dead links
    target_quotas = build_country_year_quotas(pool_df, target_images, country_alpha=country_alpha)

    selected_parts = []
    for (country, year), grp in pool_df.groupby(["country", "year"]):
        q = target_quotas.get((country, int(year)), 0)
        if q <= 0:
            continue
        n_cand = min(len(grp), int(math.ceil(q * oversample_factor)))
        part = grp.sample(n=n_cand, random_state=42).copy()
        part["bucket_target"] = q
        selected_parts.append(part)

    if not selected_parts:
        return pd.DataFrame(), {}

    balanced_meta_df = pd.concat(selected_parts, ignore_index=True)
    balanced_meta_df.to_csv(meta_path, index=False)
    print(f"Selected {len(balanced_meta_df):,} candidate rows (target={sum(target_quotas.values()):,} images) "
          f"across {balanced_meta_df['country'].nunique()} countries -> {meta_path}")
    return balanced_meta_df, target_quotas

yfcc_df, yfcc_quotas = stream_and_balance_yfcc15m()
display(yfcc_df.head())

YFCC15M valid rows (1900-1999):   0%|          | 0/30000 [00:00<?, ?it/s]

Loading formatted geocoded file...


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/mehdidc/yfcc15m/resolve/7ef74d0624efc36c16fe3b82aa5438f7c7260706/yfcc15m.csv
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/mehdidc/yfcc15m/resolve/7ef74d0624efc36c16fe3b82aa5438f7c7260706/yfcc15m.csv
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/mehdidc/yfcc15m/resolve/7ef74d0624efc36c16fe3b82aa5438f7c7260706/yfcc15m.csv
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/mehdidc/yfcc15m/resolve/7ef74d0624efc36c16fe3b82aa5438f7c7260706/yfcc15m.csv
Retrying in 1s [Retry 1/5].


Scanned 5,888,071 rows -> 30,000 valid pre-2000 geotagged+dated rows (1900-1999) across 161 countries.
Selected 18,795 candidate rows (target=15,000 images) across 161 countries -> d:\VSCodeProjects\CS441\data\yfcc15m_pre2000\metadata_filtered.csv


,photoid,year,datetaken,latitude,longitude,accuracy,downloadurl,title,source,country,bucket_target
0,2633650304,1980,1980-01-01 00:00:39,25.269458,55.299854,13,http://farm4.staticflickr.com/3172/2633650304_...,Dubai+Museum,yfcc15m_pre2000,AE,1
1,55343071,1991,1991-10-07 12:00:00,25.253235,55.321998,11,http://farm1.staticflickr.com/32/55343071_87a3...,HCT+Colleague,yfcc15m_pre2000,AE,2
2,55343016,1991,1991-11-01 12:00:00,25.254477,55.293502,11,http://farm1.staticflickr.com/29/55343016_27a5...,Dubai+Creek+-+Water+Taxis,yfcc15m_pre2000,AE,2
3,55343049,1992,1992-11-01 12:00:00,25.242211,55.306892,11,http://farm1.staticflickr.com/27/55343049_d3a6...,Dubai+Creek,yfcc15m_pre2000,AE,1
4,56734429,1994,1994-03-01 20:00:00,25.210241,55.253076,14,http://farm1.staticflickr.com/30/56734429_86dd...,An+HCT+Staff+Party,yfcc15m_pre2000,AE,1


In [5]:
from io import BytesIO
from collections import defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
import httpx
from PIL import Image
from tqdm.auto import tqdm

def download_yfcc_images_balanced(
    meta_df: pd.DataFrame,
    target_quotas: dict[tuple[str, int], int] | None = None,
    output_dir: Path = YFCC_DIR,
    workers: int = YFCC_DOWNLOAD_WORKERS,
    timeout: float = 10.0,
    checkpoint_every: int = 250,
) -> pd.DataFrame:
    if meta_df is None or len(meta_df) == 0:
        print("No YFCC15M pre-2000 candidates to download.")
        return pd.DataFrame()

    img_dir = output_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    out_csv = output_dir / "metadata_downloaded.csv"
    headers = {"User-Agent": "TimeguessrAI-CS441-DatasetBuilder/0.1"}

    if target_quotas is None:
        target_quotas = {
            (str(c), int(y)): int(t)
            for (c, y), t in meta_df.groupby(["country", "year"])["bucket_target"].first().items()
        }

    # Resume from existing metadata_downloaded.csv if present
    filled: dict[tuple[str, int], int] = defaultdict(int)
    valid_records: list[dict] = []
    seen_pids: set[str] = set()
    if out_csv.exists():
        prev_df = pd.read_csv(out_csv, dtype={"photoid": str})
        valid_records = prev_df.to_dict(orient="records")
        for r in valid_records:
            seen_pids.add(str(r["photoid"]))
            filled[(str(r["country"]), int(r["year"]))] += 1
        print(f"Resuming YFCC15M downloads from {len(valid_records):,} existing images in {out_csv}")

    # Organize remaining candidates into per-(country, year) queues
    queues: dict[tuple[str, int], deque] = {}
    for (country, year), grp in meta_df.groupby(["country", "year"]):
        key = (str(country), int(year))
        if target_quotas.get(key, 0) > filled[key]:
            rem = [r for r in grp.to_dict(orient="records") if str(r["photoid"]) not in seen_pids]
            if rem:
                queues[key] = deque(rem)

    def _fetch_one(row_dict: dict) -> dict | None:
        pid = str(row_dict["photoid"])
        out_path = img_dir / f"{pid}.jpg"
        if out_path.exists():
            row_dict["image_path"] = str(out_path.relative_to(PROJECT_ROOT))
            return row_dict
        try:
            with httpx.Client(timeout=timeout, follow_redirects=True, headers=headers) as client:
                resp = client.get(row_dict["downloadurl"])
                if resp.status_code != 200 or not resp.content:
                    return None
                with Image.open(BytesIO(resp.content)) as im:
                    im.verify()
                with Image.open(BytesIO(resp.content)) as im:
                    im.convert("RGB").save(out_path, format="JPEG", quality=92)
            row_dict["image_path"] = str(out_path.relative_to(PROJECT_ROOT))
            return row_dict
        except Exception:
            return None

    total_target = sum(target_quotas.values())
    pbar = tqdm(total=total_target, initial=len(valid_records), desc="Downloading balanced YFCC15M images")
    since_checkpoint = 0

    active_keys = list(queues.keys())
    try:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            while active_keys:
                batch_rows = []
                next_active = []
                for key in active_keys:
                    need = target_quotas[key] - filled[key]
                    if need <= 0 or not queues[key]:
                        continue
                    batch_rows.append((key, queues[key].popleft()))
                    if queues[key]:
                        next_active.append(key)

                if not batch_rows:
                    break

                futures = {ex.submit(_fetch_one, row): key for key, row in batch_rows}
                for fut in as_completed(futures):
                    key = futures[fut]
                    res = fut.result()
                    if res is not None and filled[key] < target_quotas[key]:
                        filled[key] += 1
                        seen_pids.add(str(res["photoid"]))
                        valid_records.append(res)
                        pbar.update(1)
                        since_checkpoint += 1

                if since_checkpoint >= checkpoint_every:
                    pd.DataFrame(valid_records).to_csv(out_csv, index=False)
                    since_checkpoint = 0

                active_keys = [k for k in next_active if filled[k] < target_quotas[k] and queues[k]]

    except KeyboardInterrupt:
        print("\nInterrupted! Saving downloaded YFCC15M metadata so far...")
    finally:
        pbar.close()
        downloaded_df = pd.DataFrame(valid_records)
        if len(downloaded_df) > 0:
            downloaded_df.to_csv(out_csv, index=False)
            print(f"Saved {len(downloaded_df):,} / {total_target:,} target YFCC15M images "
                  f"across {downloaded_df['country'].nunique()} countries -> {out_csv}")
        else:
            print("No YFCC15M images downloaded yet.")

    return downloaded_df

yfcc_downloaded_df = download_yfcc_images_balanced(yfcc_df, yfcc_quotas)
display(yfcc_downloaded_df.head())

Saved 13,814 / 15,000 target YFCC15M images across 157 countries -> d:\VSCodeProjects\CS441\data\yfcc15m_pre2000\metadata_downloaded.csv


,photoid,year,datetaken,latitude,longitude,accuracy,downloadurl,title,source,country,bucket_target,image_path
0,56737139,1995,1995-01-01 12:00:00,25.253235,55.321998,11,http://farm1.staticflickr.com/31/56737139_9242...,An+HCT+Foundations+Group,yfcc15m_pre2000,AE,1,data\yfcc15m_pre2000\images\56737139.jpg
1,56734429,1994,1994-03-01 20:00:00,25.210241,55.253076,14,http://farm1.staticflickr.com/30/56734429_86dd...,An+HCT+Staff+Party,yfcc15m_pre2000,AE,1,data\yfcc15m_pre2000\images\56734429.jpg
2,55343049,1992,1992-11-01 12:00:00,25.242211,55.306892,11,http://farm1.staticflickr.com/27/55343049_d3a6...,Dubai+Creek,yfcc15m_pre2000,AE,1,data\yfcc15m_pre2000\images\55343049.jpg
3,8695186,1998,1998-10-01 00:00:00,25.267595,55.323371,12,http://farm1.staticflickr.com/8/8695186_dbea8f...,Funky+Shop+at+City+Centre+DXB,yfcc15m_pre2000,AE,1,data\yfcc15m_pre2000\images\8695186.jpg
4,55343071,1991,1991-10-07 12:00:00,25.253235,55.321998,11,http://farm1.staticflickr.com/32/55343071_87a3...,HCT+Colleague,yfcc15m_pre2000,AE,2,data\yfcc15m_pre2000\images\55343071.jpg
